# 04 EBBA 检测机制（Tier B，缩比）

本 Notebook 检验论文的检测主张在容器可承受的规模下是否成立。所有结论都是 **proxy**：

- 数据：CIFAR-10 固定子集 20,000 训练 / 2,000 测试；任务无关图像用 CIFAR-100 测试集 5,000 张代理；
- 模型：SmallCNN（约 1.2M 参数）；
- 攻击：BadNets（3×3 白块）与 Blend（固定图案，α=0.2），目标类 0，投毒率 10%；
- 训练：SGD，30 epochs，seed 2024，带轻度损坏增广（噪声/模糊/对比度）。

**阈值说明**：缩比实验用 `λ_proxy = 2.2` 而非论文的 3.0。A20 已证明 $K=10$ 时
$\max_k(E_k-\mu)/\sigma \le \sqrt{K-1}=3$ 且只能取等号，因此 Eq.8 的严格 `>3σ` 在此不可达。
2.2 取自本代理实测的 clean z≈1.77 与后门 z≈2.66 之间；这是代理标定，不是论文设定。

按项目规则，proxy 成功只能记 `PARTIAL`，失败记 `CONTRADICTED`。

## 检测流程

1. 对任务无关图像做前向，得到伪标签；
2. 每个伪标签类别保留最多 50 张（均衡选择）；
3. 对每条样本执行 Eq.3 的 clamp，再算 Eq.4-6 的标签能量；
4. 计算 $\mu,\sigma$，检查 Eq.8 是否在某个标签上成立。


In [ ]:
import json
import os
import sys
from pathlib import Path

# 定位仓库根目录：容器内是 /app，本地打开时可能是 notebooks/ 或其子目录。
ROOT = Path(os.environ.get("EBBA_ROOT", Path.cwd())).resolve()
if not (ROOT / "src" / "ebba").exists():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "ebba").exists():
            ROOT = cand
            break
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("EBBA_ROOT", str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from ebba import config, nbutil
from ebba.claims import load_registry, run, run_many

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("project root:", ROOT)
print("artifacts   :", config.ARTIFACTS_DIR)
print("reports     :", config.REPORTS_DIR)
print("fast mode   :", config.FAST)


## B01：目标标签是否在 top 输出中更常见？（Fig.1）

后门模型会把无关图像也推向目标类。统计目标标签在 5,000 张 OOD 图像输出中的排名：
若后门模型的 top-3 频率显著高于均匀基线 0.30，且干净模型没有同样偏移，则机制成立。


In [ ]:
r = nbutil.show("B01", max_measured_chars=2600)
for name, rec in r.measured["models"].items():
    if isinstance(rec, dict):
        print(f'{name:>8}: mean_rank={rec["mean_rank"]:.2f}, top1={rec["top1"]:.3f}, top3={rec["top3"]:.3f}')


## B02：Eq.8 能否在无任务样本时分开后门模型与干净模型？

对每个模型取均衡 OOD 子集，算 10 个标签能量，报告最大偏离、margin 与检测结果。
判据：后门模型 `margin > 0` 且目标标签 = 0；干净模型 `margin < 0`。
由于 $K=10$ 时 $z\le3$，此处用代理阈值 `λ_proxy=2.2`。
若只有部分攻击被正确检出，结论记 `PARTIAL`（机制存在但不普适）；全部漏检才记 `CONTRADICTED`。


In [ ]:
r = nbutil.show("B02", max_measured_chars=3000)
for name, rec in r.measured["models"].items():
    print(f'{name:>8}: target={rec["target"]}, margin={rec["margin"]:+.3f}, '
          f'z={rec["zscore"]:.2f}, backdoored={rec["is_backdoored"]}')


In [ ]:
import numpy as np

models = r.measured["models"]
fig, ax = plt.subplots(figsize=(7, 3.2))
for name, rec in models.items():
    ax.bar(np.arange(10) + (0.2 if name == "badnets" else -0.2 if name == "blend" else 0),
           np.asarray(rec["energy"]) - np.mean(rec["energy"]),
           width=0.2, label=name, alpha=0.85)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("label")
ax.set_ylabel("E_k - mean(E)")
ax.set_title("Label energy deviation on task-agnostic proxy images")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## B03：λ 的可用区间与 Fig.80

固定后门模型的目标标签，扫描 λ，要求“后门 margin > 0 且干净 margin < 0”。
论文报告可用区间约为 [1.8, 5.7]。缩比模型只需与之有重叠，且 `λ_proxy=2.2` 落在可用区间内；
区间端点由模型决定，不能直接对齐论文。`λ=3` 的可行性由 A20 单独判定。

Fig.80 是单模型图，故任一攻击复现出重叠区间即通过；Blend 在任何 λ 下都检不出，
那是 B02 的结论，不在 B03 重复计为“区间形状不符”。


In [ ]:
r = nbutil.show("B03")
for attack, rec in r.measured["attacks"].items():
    print(f'{attack}: valid={rec["valid_interval"]}, chosen_valid={rec["chosen_valid"]}')


## B11：均衡选择是否减少误报？（Fig.78/79）

论文强调“每类保留 N 张”而不是随便取一批图。消融方式：

- 均衡：每个伪标签最多 50 张；
- 不均衡：只保留伪标签最集中的 3 类，共 500 张。

比较目标标签相对最强非目标标签的领先量 $z_t-\max_{k\ne t}z_k$，并统计两种选择下被 Eq.8
误标的非目标标签数。论文称不均衡选择会让其他标签的能量接近目标，故均衡选择的领先量应更大且为正。


In [ ]:
r = nbutil.show("B11", max_measured_chars=2600)
for name, rec in r.measured["models"].items():
    print(f'{name:>8}: lead balanced={rec["balanced_lead"]:+.2f}, '
          f'imbalanced={rec["imbalanced_lead"]:+.2f}, '
          f'false flags {rec["balanced_false_flags"]}/{rec["imbalanced_false_flags"]}')


## B12：自适应软标签攻击与双边检验（机制演示）

论文的 Eq.8 是单边检验：只看“能量偏高”。如果攻击者把目标能量压到均值以下，
单边检验会漏检。附录 G.4 讨论了这一点。

这里**不训练自适应攻击**（那属于 Tier C），而是在实测能量向量上把目标能量置为
$\mu-4\sigma$，比较单边与双边 $|E-\mu|>\lambda\sigma$ 的行为。
它只说明检验统计量的性质，不能当作对论文 13 组自适应实验的复现。


In [ ]:
r = nbutil.show("B12", max_measured_chars=2600)
print("one-sided detected:", r.measured["one_sided"]["is_backdoored"])
print("two-sided detected:", r.measured["two_sided"]["is_backdoored"],
      "target:", r.measured["two_sided"]["target"])


## 本 Notebook 的小结

- B01-B03 检验“检测机制”是否按论文方向工作；
- B11 检验均衡选择这一实现细节；
- B12 只是检验统计量的机制演示，不代表自适应攻击已被复现。

若某个 B 声明为 `CONTRADICTED`，需要区分两种可能：方法在缩比设定下确实不成立，
或缩比模型的训练/攻击强度不足。manifest 中的模型精度与 ASR 门槛用于排除后一种解释，
但缩比模型与论文 ResNet18 的差距始终是结论的适用边界。
